In [22]:
import pandas as pd
import pickle
import os
from collections import Counter
import re

In [23]:
!ls

Figure6A.ipynb				  hpvJunction-Copy1.ipynb
Figure6A.matrix_06_11_25.csv		  HPV_junctions-Copy1.ipynb
hpvBreakpoints_for_12_models-Copy1.ipynb  Sup_6_02_13_24.xlsx
HPV_heatmap_all_sample-Copy1.ipynb


In [24]:
#read from sup6
clinicInfo = pd.read_excel('Sup_6_02_13_24.xlsx')

In [25]:
geneStruct = {}
gtfFileGenome = '/home/wenjingu/remillsscr/HPV_fusion/ref_hg19/Homo_sapiens.GRCh38.105.transcript.gtf'
with open(gtfFileGenome) as genome:
    rows = genome.read().rstrip().split('\n')
    for eachRow in rows:
        hasGeneName = False
        for each in eachRow.split('\t')[8].split(';'):
            if each.split('"')[0] == ' gene_name ':
                geneName = each.split('"')[1]
                hasGeneName = True
            if each.split('"')[0] == ' transcript_id ':
                trID = each.split('"')[1]
        if not hasGeneName:
            geneName = trID
        chro = eachRow.split('\t')[0]
        start = eachRow.split('\t')[3]
        end = eachRow.split('\t')[4]
        if trID not in geneStruct:
            geneStruct[trID] = [geneName,chro,start,end]
        else:
            print(eachRow)

In [26]:
geneStruct

{'ENST00000379236': ['TNFRSF4', '1', '1211340', '1214153'],
 'ENST00000497869': ['TNFRSF4', '1', '1211340', '1214138'],
 'ENST00000453580': ['TNFRSF4', '1', '1212019', '1213498'],
 'ENST00000328596': ['TNFRSF18', '1', '1203508', '1206571'],
 'ENST00000379268': ['TNFRSF18', '1', '1203508', '1206592'],
 'ENST00000486728': ['TNFRSF18', '1', '1203844', '1205680'],
 'ENST00000379265': ['TNFRSF18', '1', '1203844', '1206571'],
 'ENST00000673477': ['ATAD3B', '1', '1471765', '1497848'],
 'ENST00000472194': ['ATAD3B', '1', '1478026', '1497848'],
 'ENST00000378736': ['ATAD3B', '1', '1479049', '1482662'],
 'ENST00000485748': ['ATAD3B', '1', '1483485', '1496202'],
 'ENST00000474481': ['ATAD3B', '1', '1484569', '1496201'],
 'ENST00000308647': ['ATAD3B', '1', '1471784', '1496201'],
 'ENST00000565563': ['ENST00000565563', '1', '1249777', '1251334'],
 'ENST00000442483': ['ENST00000442483', '1', '2212523', '2220738'],
 'ENST00000416931': ['MTND1P23', '1', '629062', '629433'],
 'ENST00000428803': ['RPL7P

In [29]:
combineGeneRes = []
i = 0
for sample in clinicInfo['Sample ID']:
    sampleName = clinicInfo['Sample name'][i]
    sampleNumber = clinicInfo['Sample'][i]
    tumorType = clinicInfo['TumorType'][i]
    if pd.notna(clinicInfo['HPV integration'][i]):
        insList = clinicInfo['HPV integration'][i].split(';')
        for ins in insList:
            if ins != '':
                chro = ins.split(':')[0].replace('chr','')
                site = ins.split(':')[1]
                farFromGene = True
                for gene in geneStruct:
                    geneName = geneStruct[gene][0]
                    AroundGene = False
                    InGene = False
    #                 InExon = False
                    chro1 = geneStruct[gene][1]
                    start = geneStruct[gene][2]
                    end = geneStruct[gene][3]
                    if chro == chro1 and int(site) >= int(start)-20000 and int(site) <= int(end) + 20000:
                        AroundGene = True
                        farFromGene = False
                    if chro == chro1 and int(site) >= int(start) and int(site) <= int(end):
                        InGene = True
    #                     exonStartList = geneStruct[gene][4].split(',')
    #                     exonEndList = geneStruct[gene][5].split(',')
    #                     i = 0
    #                     #print(exonStartList)
    #                     for eachStart in exonStartList:
    #                         if eachStart != '':
    #                             if int(site) >= int(eachStart) and int(site) <= int(exonEndList[i]):
    #                                 InExon = True
    #                             i += 1
                    if AroundGene == True:
                        combineGeneRes.append([sample,sampleName,sampleNumber,tumorType,ins,geneName,AroundGene,InGene])
                if farFromGene == True:
                    AroundGene = False
                    InGene = False
                    geneName = 'intergenic'
                    combineGeneRes.append([sample,sampleName,sampleNumber,tumorType,ins,geneName,AroundGene,InGene])
    i += 1
                    



In [57]:
combineGeneResDf = pd.DataFrame(combineGeneRes,columns = ['sample','sampleName','sampleNumber','tumorType','ins','gene','aroundGene','inGene'])

In [58]:
combineGeneResDf = combineGeneResDf.drop_duplicates().reset_index(drop=True)

In [59]:
countIns = {}
i = 0
for sample in combineGeneResDf['sample']:
    gene = combineGeneResDf['gene'][i]
    if gene != 'intergenic':
        key = sample + '.' + gene 
        if key not in countIns:
            countIns[key] = 1
        else:
            countIns[key] += 1
    i += 1

In [89]:
plot_combine_Df = combineGeneResDf[combineGeneResDf['aroundGene'] == True].iloc[:,0:6]

In [168]:
plot_combine_Df

,sample,sampleName,sampleNumber,tumorType,ins,gene
0,2920-CB-2,SOP-006-A,SOP-006LR1,LR,chr8:3220248,CSMD1
2,2920-CB-6,SOP-011-A,SOP-011LR1,LR,chr4:165363130,MSMO1
3,2920-CB-6,SOP-011-A,SOP-011LR1,LR,chr4:165363130,CPE
4,2920-CB-6,SOP-011-A,SOP-011LR1,LR,chr4:165363130,CPE
6,2920-CB-6,SOP-011-A,SOP-011LR1,LR,chr9:5890106,KIAA2026
...,...,...,...,...,...,...
3964,2789-CB-89,SOP-075-C,SOP-075LR3,LR,chr12:6392812,SCNN1A
3965,2789-CB-89,SOP-075-C,SOP-075LR3,LR,chr12:6392812,LTBR
3966,2789-CB-89,SOP-075-C,SOP-075LR3,LR,chr12:6392812,ENST00000663143
3967,2789-CB-89,SOP-075-C,SOP-075LR3,LR,chr12:6392812,ENST00000541888


In [90]:
#creat heatmap matrix include all the samples
heat_matrix = []
sampleCount = 0
sampleList = []
i = 0 
for sample in plot_combine_Df['sample']:
    sampleList.append(plot_combine_Df['sampleNumber'][i])
    sampleCount += 1
    geneListCount = []
    for gene in set(plot_combine_Df['gene']):
        key = sample + '.' + gene
        if key in countIns:
            geneListCount.append(countIns[key])
        else:
            geneListCount.append(0)
    heat_matrix.append(geneListCount)


In [169]:
matrix_df = pd.DataFrame(heat_matrix,index=plot_combine_Df['sampleNumber'],columns=set(plot_combine_Df['gene'])).drop_duplicates()

In [171]:
matrix_df.to_csv('Figure6A.matrix_06_11_25.csv')

In [172]:
#plot version, remove transcripts with no gene name
matrix_plot_df = matrix_df.loc[:,[each for each in matrix_df.columns if 'ENST' not in each]]

In [173]:
matrix_plot_df = matrix_plot_df.drop_duplicates()

In [174]:
matrix_plot_df.loc[:, matrix_plot_df.sum(axis=0) > 2].to_csv('Figure6A.plot.matrix_06_18_25.csv')

In [175]:
#create sample info
plot_clinic_df = clinicInfo[['Sample ID','Sample name','TumorType','HPV type','Sample']]

In [176]:
plot_clinic_df['patientID'] = ['-'.join(each.split('-')[0:2]) for each in plot_clinic_df['Sample name']]

/tmp/ipykernel_3965035/3414126619.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  plot_clinic_df['patientID'] = ['-'.join(each.split('-')[0:2]) for each in plot_clinic_df['Sample name']]


In [177]:
#label the HPV type in same patient to fill the missing value
HPV_type_info = plot_clinic_df[['patientID','HPV type']].dropna().drop_duplicates()

In [178]:
plot_clinic_df = plot_clinic_df.merge(HPV_type_info,on = 'patientID',suffixes = ('_x',''))

In [179]:
plot_clinic_df = plot_clinic_df.drop(['HPV type_x','Sample name'],axis=1)
plot_clinic_df = plot_clinic_df.rename(columns = {'HPV type':'HPVType','Sample ID':'SampleID'})

In [180]:
plot_clinic_df.to_csv('Figure6A.plot.info_06_18_25.csv')